# MWE Filter Evaluation Analysis (HIMYM s04e12 l1055-1259)

## Introduction
This notebook compares phrasal-verb filter strategies on the same gold slice and shows overall metrics plus exact TP/FP/FN examples per filter.

## What Each Filter Means
Filter behavior is implemented in `app/search/mwe/phrasal_verbs.py` (`PV_FILTERS`).

- **none**: no filtering; accepts every lemma match (baseline).
- **dep_only**: strict dependency filter; keeps only matches where spaCy finds a `prt` particle attached to the verb.
- **blocklist**: rule-based blocklist; rejects known high-FP literal combinations (e.g., `go to`, `be in`) and keeps everything else.
- **dep_extended**: accepts `prt` and `advmod`; for `prep`, rejects cases with `pobj` when the verb is high-frequency, otherwise accepts.
- **dep_highfreq**: applies strict `prt` filtering only for high-frequency verbs; non-high-frequency verbs are accepted without dep filtering.
- **hybrid**: blocklist-first plus dependency-aware fallback; blocklisted pairs pass only if true `prt` is found, non-blocklisted pairs are mostly permissive (accepts `prt`/`advmod`/`prep`, with permissive fallback).

## Output Sections
1. Metrics comparison table across filters.
2. Per-filter breakdown: False Positives, False Negatives, and True Positives.

In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 220)

In [2]:
DATASET_DIR = Path('../manual_tests/datasets/himym/s04e12_benefits_l1055_1259')
RUNS_DIR = DATASET_DIR / 'runs'
GOLD_PATH = DATASET_DIR / 'gold.csv'

PREFERRED_ORDER = ['none', 'dep_only', 'blocklist', 'dep_extended', 'dep_highfreq', 'hybrid']

def extract_filter_name(path: Path) -> str | None:
    m = re.search(r'_filter_([^_]+(?:_[^_]+)*)_predictions\.csv$', path.name)
    return m.group(1) if m else None

def to_lookup(df: pd.DataFrame) -> dict:
    lookup = {}
    for _, row in df.iterrows():
        lookup[row['key']] = row.to_dict()
    return lookup

gold_df = pd.read_csv(GOLD_PATH)
gold_df['line_index'] = gold_df['line_index'].astype(int)
gold_df['key'] = list(zip(gold_df['line_index'], gold_df['expression_type'], gold_df['expression']))
gold_df = gold_df.drop_duplicates(subset=['key']).reset_index(drop=True)
gold_lookup = to_lookup(gold_df)
gold_keys = set(gold_lookup.keys())

prediction_files = sorted(RUNS_DIR.glob('*_filter_*_predictions.csv'))
filters = {}
for pred_path in prediction_files:
    name = extract_filter_name(pred_path)
    if not name:
        continue
    pred_df = pd.read_csv(pred_path)
    pred_df['line_index'] = pred_df['line_index'].astype(int)
    pred_df['key'] = list(zip(pred_df['line_index'], pred_df['expression_type'], pred_df['expression']))
    pred_df = pred_df.drop_duplicates(subset=['key']).reset_index(drop=True)
    filters[name] = pred_df

ordered_filters = [f for f in PREFERRED_ORDER if f in filters] + [f for f in filters if f not in PREFERRED_ORDER]
print('Loaded filters:', ordered_filters)
print('Gold instances:', len(gold_keys))

Loaded filters: ['none', 'dep_only', 'blocklist', 'dep_extended', 'dep_highfreq', 'hybrid']
Gold instances: 27


In [3]:
def details_df(keys, source_lookup):
    if not keys:
        return pd.DataFrame(columns=['line_index', 'expression_type', 'expression', 'speaker', 'line'])

    rows = []
    for k in sorted(keys):
        row = source_lookup[k]
        rows.append({
            'line_index': int(row['line_index']),
            'expression_type': row['expression_type'],
            'expression': row['expression'],
            'speaker': row.get('speaker', ''),
            'line': row.get('line', ''),
        })

    return pd.DataFrame(rows).sort_values(['line_index', 'expression_type', 'expression']).reset_index(drop=True)

results = {}
metric_rows = []

for filter_name in ordered_filters:
    pred_df = filters[filter_name]
    pred_lookup = to_lookup(pred_df)
    pred_keys = set(pred_lookup.keys())

    tp = gold_keys & pred_keys
    fp = pred_keys - gold_keys
    fn = gold_keys - pred_keys

    precision = len(tp) / len(pred_keys) if pred_keys else 0.0
    recall = len(tp) / len(gold_keys) if gold_keys else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    metric_rows.append({
        'Filter': filter_name,
        'Pred': len(pred_keys),
        'TP': len(tp),
        'FP': len(fp),
        'FN': len(fn),
        'Precision': round(precision, 3),
        'Recall': round(recall, 3),
        'F1': round(f1, 3),
    })

    results[filter_name] = {
        'tp': details_df(tp, gold_lookup),
        'fp': details_df(fp, pred_lookup),
        'fn': details_df(fn, gold_lookup),
    }

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

,Filter,Pred,TP,FP,FN,Precision,Recall,F1
0,none,75,25,50,2,0.333,0.926,0.490
1,dep_only,24,19,5,8,0.792,0.704,0.745
2,blocklist,71,25,46,2,0.352,0.926,0.510
3,dep_extended,35,23,12,4,0.657,0.852,0.742
4,dep_highfreq,41,22,19,5,0.537,0.815,0.647
5,hybrid,71,25,46,2,0.352,0.926,0.510


In [4]:
for filter_name in ordered_filters:
    print(f'\n=== FILTER: {filter_name} ===')

    filter_metrics = metrics_df[metrics_df['Filter'] == filter_name].reset_index(drop=True)
    print('Metrics')
    display(filter_metrics)

    print('False Positives')
    display(results[filter_name]['fp'])

    print('False Negatives')
    display(results[filter_name]['fn'])

    print('True Positives')
    display(results[filter_name]['tp'])


=== FILTER: none ===
Metrics


,Filter,Pred,TP,FP,FN,Precision,Recall,F1
0,none,75,25,50,2,0.333,0.926,0.49


False Positives


,line_index,expression_type,expression,speaker,line
0,1068,phrasal_verb,go it,Robin,"No, I went there yesterday... Stop it! Stop it! My God, what happens? When we were a couple, we lived together and we almost went insane.\n"
1,1072,phrasal_verb,boil down to,Barney,"I explained. I said, Madeline, every international conflict essentially boils down to sexual tension.\n"
2,1091,phrasal_verb,have to,Marshall,I was working and I had to take a leap here...reading this magazine. In... the room there.\n
3,1091,phrasal_verb,read in,Marshall,I was working and I had to take a leap here...reading this magazine. In... the room there.\n
4,1093,phrasal_verb,be to,Robin,"If this is a problem. You've done all the way here to read a magazine? I am willing to bet that there is a place to read this magazine at work. You know, a room with a little man on the door?\n"
5,1093,phrasal_verb,have do,Robin,"If this is a problem. You've done all the way here to read a magazine? I am willing to bet that there is a place to read this magazine at work. You know, a room with a little man on the door?\n"
6,1116,phrasal_verb,sex up,Ted,Absolutely! Let's multitasking. Use sex to spice up the boring activities.\n
7,1116,phrasal_verb,use to,Ted,Absolutely! Let's multitasking. Use sex to spice up the boring activities.\n
8,1116,phrasal_verb,use up,Ted,Absolutely! Let's multitasking. Use sex to spice up the boring activities.\n
9,1116,phrasal_verb,used to,Ted,Absolutely! Let's multitasking. Use sex to spice up the boring activities.\n


False Negatives


...

,line_index,expression_type,expression,speaker,line
0,1068,phrasal_verb,go it,Robin,"No, I went there yesterday... Stop it! Stop it! My God, what happens? When we were a couple, we lived together and we almost went insane.\n"
1,1072,phrasal_verb,boil down to,Barney,"I explained. I said, Madeline, every international conflict essentially boils down to sexual tension.\n"
2,1091,phrasal_verb,have to,Marshall,I was working and I had to take a leap here...reading this magazine. In... the room there.\n
3,1091,phrasal_verb,read in,Marshall,I was working and I had to take a leap here...reading this magazine. In... the room there.\n
4,1093,phrasal_verb,be to,Robin,"If this is a problem. You've done all the way here to read a magazine? I am willing to bet that there is a place to read this magazine at work. You know, a room with a little man on the door?\n"
5,1093,phrasal_verb,have do,Robin,"If this is a problem. You've done all the way here to read a magazine? I am willing to bet that there is a place to read this magazine at work. You know, a room with a little man on the door?\n"
6,1116,phrasal_verb,sex up,Ted,Absolutely! Let's multitasking. Use sex to spice up the boring activities.\n
7,1116,phrasal_verb,use to,Ted,Absolutely! Let's multitasking. Use sex to spice up the boring activities.\n
8,1116,phrasal_verb,use up,Ted,Absolutely! Let's multitasking. Use sex to spice up the boring activities.\n
9,1116,phrasal_verb,used to,Ted,Absolutely! Let's multitasking. Use sex to spice up the boring activities.\n


False Negatives


,line_index,expression_type,expression,speaker,line
0,1087,phrasal_verb,back together,Marshall,"You do not care what, guys? You are back together?"
1,1204,phrasal_verb,go well,Marshall,"Everything was going well. I felt more and more comfortable, more confident. I could conquer the world. One morning I'm in the eighth with a magazine."


True Positives


,line_index,expression_type,expression,speaker,line
0,1061,phrasal_verb,take out,Ted,"So, take out the trash."
1,1068,phrasal_verb,live together,Robin,"No, I went there yesterday... Stop it! Stop it! My God, what happens? When we were a couple, we lived together and we almost went insane."
2,1072,phrasal_verb,boil down,Barney,"I explained. I said, Madeline, every international conflict essentially boils down to sexual tension."
3,1091,idiom,take a leap,Marshall,I was working and I had to take a leap here...reading this magazine. In... the room there.
4,1093,idiom,all the way,Robin,"If this is a problem. You've done all the way here to read a magazine? I am willing to bet that there is a place to read this magazine at work. You know, a room with a little man on the door?"
5,1110,phrasal_verb,end in,Lily,"No. It could ruin your friendship. When two former try the ""right opportunity"", someone always ends in pain."
6,1116,phrasal_verb,spice up,Ted,Absolutely! Let's multitasking. Use sex to spice up the boring activities.
7,1131,idiom,see a psychiatrist,Lily,You must learn to express your feelings. Perhaps you should see a psychiatrist.
8,1164,phrasal_verb,get out,Lily,"No, it's wrong. You must learn to get it out. As we did in my kindergarten class. ""The time for emotions"", every Tuesday morning."
9,1169,phrasal_verb,lie around,Robin,"Sure, it was a good one. Personal memo: leave it lying around the pizza box more often."
